# TSGC: Temporal Semantic Gradient Compression
### Benchmarking against naive alternatives for AI conversation context storage

**Author:** Utkarsh Aggarwal  
**Project:** UniMemo — AI Context Bridge  
**GitHub:** https://github.com/Utkarsh-Aggarwal/UniMemo

---

### What we're testing

When storing AI conversation history in a resource-constrained environment (Chrome extension, 10MB limit),
how does **TSGC** compare to simpler alternatives?

**Methods compared:**
1. `RAW` — Store everything as-is
2. `TRUNCATE` — Keep only last N messages
3. `UNIFORM` — Compress every message equally (50% of sentences)
4. `DEFLATE_ONLY` — Raw DEFLATE, no semantic pruning
5. `TSGC` — Temporal Semantic Gradient Compression (our method)

**Metrics:**
- Compression ratio (storage savings)
- Key term retention (semantic fidelity)
- Recency fidelity (are recent messages preserved?)
- Information density of output

In [ ]:
# Install dependencies
!pip install -q rouge-score matplotlib seaborn pandas numpy

In [ ]:
import re
import zlib
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from collections import Counter
from rouge_score import rouge_scorer

# Styling
plt.rcParams.update({
    'figure.facecolor': '#0d0f14',
    'axes.facecolor':   '#161921',
    'axes.edgecolor':   '#2a2d3a',
    'text.color':       '#f0f2f8',
    'axes.labelcolor':  '#8b90a8',
    'xtick.color':      '#8b90a8',
    'ytick.color':      '#8b90a8',
    'grid.color':       '#1e2330',
    'grid.linestyle':   '--',
    'axes.titlecolor':  '#f0f2f8',
    'font.family':      'DejaVu Sans',
    'axes.titlesize':   14,
    'axes.labelsize':   11,
})

ACCENT   = '#7c6fe0'
SUCCESS  = '#22c55e'
WARNING  = '#f59e0b'
DANGER   = '#ef4444'
MUTED    = '#4e5368'

print('✅ Setup complete')

## 1. Synthetic Conversation Dataset
We generate realistic AI conversations with:
- Mixed message lengths (short questions, long explanations)
- Deliberate repetition (AI restating earlier context)
- Filler phrases (common in assistant responses)
- Technical content (keywords that should be preserved)

In [ ]:
CONVERSATIONS = [
    # Conv 1 — Debugging session
    [
        {'role':'user', 'content':'I have a Python function that should return sorted unique elements but it returns duplicates. Here is the code: def get_items(lst): return sorted(lst)'},
        {'role':'assistant', 'content':'Sure! I can help with that. The issue is that sorted() preserves duplicates — it only sorts, it does not remove them. The solution is to convert to a set first. Here is the corrected function: def get_items(lst): return sorted(set(lst)). The set() call removes duplicates because sets only store unique values. Then sorted() puts them in order.'},
        {'role':'user', 'content':'Great that works! Now I need to make it case insensitive for strings.'},
        {'role':'assistant', 'content':'Of course! To make it case insensitive, you should normalize the strings before deduplication. As I mentioned earlier, set() handles uniqueness. Now we combine it with a key function. Here is the updated version: def get_items(lst): seen = set(); result = []; [result.append(x) for x in lst if x.lower() not in seen and not seen.add(x.lower())]; return sorted(result, key=str.lower). This is important because it preserves the original casing while deduplicating case-insensitively.'},
        {'role':'user', 'content':'What if the list contains both strings and numbers?'},
        {'role':'assistant', 'content':'Good question. To summarize what we have built so far: we have a function that sorts and deduplicates. Now for mixed types, Python 3 does not allow comparing strings and integers directly in sort. The solution is to separate them. Here is the approach: def get_items(lst): nums = sorted(set(x for x in lst if isinstance(x, (int,float)))); strs = sorted(set(x.lower() for x in lst if isinstance(x, str))); return nums + strs. This is the key insight — partition by type first, then sort each group.'},
        {'role':'user', 'content':'Can you add error handling?'},
        {'role':'assistant', 'content':'Certainly! Error handling is important for production code. As I noted earlier, the function handles strings and numbers. Now let us add try-except blocks. The important thing to note is that we should catch TypeError specifically. Here is the final version with proper error handling that also logs issues.'},
        {'role':'user', 'content':'How do I write unit tests for this?'},
        {'role':'assistant', 'content':'Great question! Unit testing is a critical skill. Let me walk you through it step by step. We will use Python unittest module. The key test cases are: empty list, single element, duplicates only, mixed types, all same case strings. Each test should be independent and test one specific behavior. I will show you the complete test class now.'},
    ],
    # Conv 2 — Architecture discussion
    [
        {'role':'user', 'content':'Should I use microservices or monolith for my startup?'},
        {'role':'assistant', 'content':'This is one of the most common architecture questions. For a startup, I strongly recommend starting with a monolith. Here is why: speed of iteration matters more than scalability at day one. Microservices add significant operational overhead — you need service discovery, distributed tracing, inter-service authentication. A monolith lets you move fast.'},
        {'role':'user', 'content':'But what about scaling issues later?'},
        {'role':'assistant', 'content':'Of course, scaling is a real concern. However, as I mentioned, premature optimization is the root of many problems. The key insight is that you will know where to split when you actually hit bottlenecks. Most startups fail before they hit scaling issues. If you do scale, you can extract services from a well-structured monolith. The important thing is to keep your modules loosely coupled from the start — that makes the future migration possible.'},
        {'role':'user', 'content':'How do I keep a monolith well-structured?'},
        {'role':'assistant', 'content':'Great question. To summarize our discussion: we agreed on starting with a monolith. Now for structure, the key principle is domain-driven design. Organize code by business domain, not technical layer. So instead of /models /controllers /services — use /auth /payments /users each containing their own models, controllers, and services. This is the approach that makes future extraction to microservices straightforward.'},
        {'role':'user', 'content':'What database should I use?'},
        {'role':'assistant', 'content':'Database choice is critical. For most startups, PostgreSQL is the safe default. It handles 99% of use cases, has excellent support, and is battle-tested. Avoid the temptation to use NoSQL unless you have a specific reason — document stores like MongoDB seem flexible but can lead to consistency problems. The important thing is that PostgreSQL can do document storage too via JSONB columns.'},
    ],
    # Conv 3 — Short Q&A (tests recency preservation)
    [
        {'role':'user', 'content':'What is a closure in JavaScript?'},
        {'role':'assistant', 'content':'A closure is a function that remembers the variables from the scope where it was created, even after that scope has finished executing. For example: function makeCounter() { let count = 0; return function() { count++; return count; }; } The inner function closes over the count variable.'},
        {'role':'user', 'content':'Give me a practical use case.'},
        {'role':'assistant', 'content':'Sure! A practical use case is creating private state. In JavaScript before classes were common, closures were the primary way to encapsulate data. Another important use case is event handlers that remember configuration — like a button that remembers which item it is associated with in a list.'},
        {'role':'user', 'content':'How does this relate to the module pattern?'},
        {'role':'assistant', 'content':'Great connection! The module pattern uses closures to create private and public interfaces. As I explained, closures remember their outer scope. The module pattern wraps everything in an IIFE and returns only what should be public. This was the standard pattern before ES6 modules. The key benefit is encapsulation without class syntax.'},
        {'role':'user', 'content':'What about memory leaks with closures?'},
        {'role':'assistant', 'content':'This is an important concern. Closures can cause memory leaks if they hold references to DOM elements that get removed. The solution is to null out references when no longer needed. The key rule is: if a closure lives longer than the data it references, you have a potential leak. Event listener cleanup is the most common place this matters.'},
    ]
]

print(f'Dataset: {len(CONVERSATIONS)} conversations')
for i, c in enumerate(CONVERSATIONS):
    total_chars = sum(len(m['content']) for m in c)
    print(f'  Conv {i+1}: {len(c)} messages, {total_chars:,} chars')

## 2. Implement All Compression Methods

In [ ]:
# ── Signal words (same as TSGC JS implementation) ─────────────
SIGNAL_WORDS = {
    'because','therefore','however','but','although','instead','error',
    'result','conclusion','solution','problem','issue','answer','key',
    'important','note','warning','step','must','should','never','always',
    'define','means','returns','function','class','approach','algorithm',
    'difference','summary','finally','specifically','example','critical',
    'insight','benefit','concern','pattern','principle'
}

FILLER_RE = [
    r'^(sure|of course|certainly|absolutely|great|okay)[!.,]?\s*',
    r'^(as i (mentioned|said|noted|explained) (earlier|above|before))[,.]?\s*',
    r'^(to (summarize|recap|reiterate))[,:]?\s*',
    r'hope (this|that) (helps|clarifies)',
    r'feel free to (ask|let me know)',
    r'^(great question|good question)[!.]?',
]

def split_sentences(text):
    sents = re.split(r'(?<=[.!?])\s+', text.strip())
    return [s.strip() for s in sents if len(s.strip()) > 10]

def is_filler(sentence):
    for pat in FILLER_RE:
        if re.search(pat, sentence.strip(), re.IGNORECASE):
            return True
    return False

def score_sentence(sentence, index, total):
    """Score a sentence by position, density, and keyword presence."""
    # Position score (U-shape — first and last matter most)
    rel_pos  = index / max(total - 1, 1)
    pos_score = 1 - abs(rel_pos - 0.5) * 0.6

    # Density score (unique word ratio)
    words    = re.findall(r'\b[a-z]{3,}\b', sentence.lower())
    dens_score = len(set(words)) / len(words) if words else 0

    # Keyword score
    kw_hits  = sum(1 for w in words if w in SIGNAL_WORDS)
    kw_score = min(1.0, kw_hits / 3)

    return 0.3 * pos_score + 0.4 * dens_score + 0.3 * kw_score

def extract_top_sentences(text, ratio, role='assistant'):
    sents = split_sentences(text)
    if len(sents) <= 1:
        return text
    if role == 'assistant':
        sents = [s for s in sents if not is_filler(s)] or sents
    keep = max(1, round(len(sents) * ratio))
    scored = [(s, i, score_sentence(s, i, len(sents))) for i, s in enumerate(sents)]
    top = sorted(scored, key=lambda x: -x[2])[:keep]
    top = sorted(top, key=lambda x: x[1])  # restore order
    return ' '.join(s for s, _, _ in top)

def deflate_ratio(text):
    """Simulate DEFLATE compression ratio on text."""
    raw  = text.encode('utf-8')
    comp = zlib.compress(raw, level=6)
    return len(comp) / len(raw)


# ── Compression Methods ────────────────────────────────────────

def method_raw(messages):
    """Baseline: store everything as-is."""
    return messages

def method_truncate(messages, keep_last=10):
    """Naive truncation: keep only the last N messages."""
    return messages[-keep_last:]

def method_uniform(messages, ratio=0.5):
    """Uniform compression: compress every message equally."""
    result = []
    for m in messages:
        content = extract_top_sentences(m['content'], ratio, m['role'])
        result.append({'role': m['role'], 'content': content})
    return result

def method_deflate_only(messages):
    """DEFLATE only — no semantic pruning. Simulated as ratio."""
    # Returns messages as-is (DEFLATE is transparent),
    # but we'll track the byte-level compression separately
    return messages

def method_tsgc(messages, z1=5, z2=15):
    """TSGC: Temporal Semantic Gradient Compression."""
    total  = len(messages)
    result = []
    seen_sents = set()

    # Layer 1: Deduplication
    deduped = []
    for m in messages:
        sents  = split_sentences(m['content'])
        unique = []
        for s in sents:
            key = s.strip().lower()
            if len(key) < 20 or key not in seen_sents:
                seen_sents.add(key)
                unique.append(s)
        content = ' '.join(unique).strip()
        if content:
            deduped.append({'role': m['role'], 'content': content})

    # Layer 2: Temporal gradient
    total_d = len(deduped)
    for idx, m in enumerate(deduped):
        recency = total_d - 1 - idx  # 0 = most recent
        if recency < z1:
            zone, ratio = 1, 1.0      # verbatim
        elif recency < z2:
            zone, ratio = 2, 0.40     # moderate pruning
        else:
            zone, ratio = 3, 0.15     # aggressive pruning

        if ratio < 1.0 and len(m['content']) > 80:
            content = extract_top_sentences(m['content'], ratio, m['role'])
        else:
            content = m['content']

        result.append({'role': m['role'], 'content': content, '_zone': zone})

    return result


print('✅ All compression methods implemented')

## 3. Evaluation Metrics

In [ ]:
scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)

def chars(messages):
    return sum(len(m['content']) for m in messages)

def storage_bytes(messages):
    return len(json.dumps([{'role':m['role'],'content':m['content']} for m in messages]).encode())

def effective_bytes(messages, with_deflate=True):
    """Bytes after optional DEFLATE on top."""
    raw = storage_bytes(messages)
    if not with_deflate:
        return raw
    text = json.dumps([{'role':m['role'],'content':m['content']} for m in messages])
    return len(zlib.compress(text.encode(), level=6))

def key_term_retention(original, compressed):
    """% of signal words from original that appear in compressed."""
    orig_text = ' '.join(m['content'] for m in original).lower()
    comp_text = ' '.join(m['content'] for m in compressed).lower()
    orig_words = set(re.findall(r'\b[a-z]{3,}\b', orig_text))
    orig_signals = orig_words & SIGNAL_WORDS
    if not orig_signals:
        return 1.0
    retained = sum(1 for w in orig_signals if w in comp_text)
    return retained / len(orig_signals)

def recency_fidelity(original, compressed, last_n=3):
    """How well are the last N messages preserved?"""
    orig_recent = ' '.join(m['content'] for m in original[-last_n:])
    comp_recent = [m for m in compressed if m.get('_zone', 1) == 1]
    if not comp_recent:
        # Use last last_n messages
        comp_recent = compressed[-last_n:]
    comp_text   = ' '.join(m['content'] for m in comp_recent[-last_n:])
    if not comp_text.strip():
        return 0.0
    scores = scorer.score(orig_recent, comp_text)
    return scores['rougeL'].fmeasure

def info_density(messages):
    """Average unique-word ratio across all messages."""
    densities = []
    for m in messages:
        words = re.findall(r'\b[a-z]{3,}\b', m['content'].lower())
        if words:
            densities.append(len(set(words)) / len(words))
    return np.mean(densities) if densities else 0

def evaluate(name, original, compressed, with_deflate=True):
    orig_bytes = effective_bytes(original, with_deflate=False)
    comp_bytes = effective_bytes(compressed, with_deflate=with_deflate)
    savings    = 1 - comp_bytes / orig_bytes
    return {
        'Method':             name,
        'Orig Bytes':         orig_bytes,
        'Comp Bytes':         comp_bytes,
        'Storage Savings %':  round(savings * 100, 1),
        'Key Term Retention': round(key_term_retention(original, compressed) * 100, 1),
        'Recency Fidelity':   round(recency_fidelity(original, compressed) * 100, 1),
        'Info Density':       round(info_density(compressed) * 100, 1),
        'Messages Kept':      len(compressed),
    }

print('✅ Metrics defined')

## 4. Run Benchmark

In [ ]:
all_results = []

for conv_idx, conv in enumerate(CONVERSATIONS):
    print(f'\n── Conversation {conv_idx + 1} ({len(conv)} messages) ──')

    methods = {
        'RAW':          (method_raw(conv),              False),
        'TRUNCATE-10':  (method_truncate(conv, 10),     False),
        'UNIFORM-50%':  (method_uniform(conv, 0.5),     False),
        'DEFLATE ONLY': (method_deflate_only(conv),     True),
        'TSGC':         (method_tsgc(conv),             True),
    }

    for name, (compressed, use_deflate) in methods.items():
        res = evaluate(name, conv, compressed, with_deflate=use_deflate)
        res['Conv'] = conv_idx + 1
        all_results.append(res)
        print(f'  {name:15} | Savings: {res["Storage Savings %"]:5.1f}% | '
              f'KeyTerms: {res["Key Term Retention"]:5.1f}% | '
              f'Recency: {res["Recency Fidelity"]:5.1f}%')

df = pd.DataFrame(all_results)

# Aggregate across conversations
agg = df.groupby('Method').agg({
    'Storage Savings %':  'mean',
    'Key Term Retention': 'mean',
    'Recency Fidelity':   'mean',
    'Info Density':       'mean',
}).round(1).reset_index()

METHOD_ORDER = ['RAW', 'TRUNCATE-10', 'UNIFORM-50%', 'DEFLATE ONLY', 'TSGC']
agg['Method'] = pd.Categorical(agg['Method'], categories=METHOD_ORDER, ordered=True)
agg = agg.sort_values('Method')

print('\n── Aggregate Results ──')
print(agg.to_string(index=False))

## 5. Graphs

In [ ]:
# ── FIGURE 1: Multi-metric comparison bar chart ────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 6))
fig.suptitle('TSGC vs Alternatives — Compression Benchmark', fontsize=16, color='#f0f2f8', fontweight='bold', y=1.02)

COLORS = {
    'RAW':          MUTED,
    'TRUNCATE-10':  DANGER,
    'UNIFORM-50%':  WARNING,
    'DEFLATE ONLY': '#4285F4',
    'TSGC':         ACCENT,
}
colors = [COLORS[m] for m in agg['Method']]

metrics = [
    ('Storage Savings %',  'Storage Savings (%)',  'Higher = better'),
    ('Key Term Retention', 'Key Term Retention (%)', 'Higher = better'),
    ('Recency Fidelity',   'Recency Fidelity (%)',   'Higher = better'),
]

for ax, (col, ylabel, subtitle) in zip(axes, metrics):
    bars = ax.bar(agg['Method'], agg[col], color=colors, width=0.6, edgecolor='none', zorder=3)
    ax.set_title(ylabel, fontsize=12, pad=10)
    ax.set_xlabel('')
    ax.set_ylabel(subtitle, fontsize=9)
    ax.tick_params(axis='x', rotation=20)
    ax.set_ylim(0, 115)
    ax.grid(axis='y', zorder=0)
    ax.spines[['top','right','left','bottom']].set_visible(False)

    # Value labels
    for bar, val in zip(bars, agg[col]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                f'{val:.1f}%', ha='center', va='bottom', fontsize=9,
                color='#f0f2f8', fontweight='bold')

    # Highlight TSGC bar
    tsgc_idx = list(agg['Method']).index('TSGC')
    bars[tsgc_idx].set_edgecolor(ACCENT)
    bars[tsgc_idx].set_linewidth(2)

plt.tight_layout()
plt.savefig('tsgc_comparison.png', dpi=150, bbox_inches='tight', facecolor='#0d0f14')
plt.show()
print('Saved: tsgc_comparison.png')

In [ ]:
# ── FIGURE 2: Radar / Spider chart ────────────────────────────
from matplotlib.patches import FancyBboxPatch

categories = ['Storage\nSavings', 'Key Term\nRetention', 'Recency\nFidelity', 'Info\nDensity']
N = len(categories)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]  # close the loop

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
fig.patch.set_facecolor('#0d0f14')
ax.set_facecolor('#161921')

RADAR_METHODS = ['TRUNCATE-10', 'UNIFORM-50%', 'DEFLATE ONLY', 'TSGC']
RADAR_COLORS  = [DANGER, WARNING, '#4285F4', ACCENT]
RADAR_ALPHA   = [0.15, 0.15, 0.15, 0.25]

for method, color, alpha in zip(RADAR_METHODS, RADAR_COLORS, RADAR_ALPHA):
    row = agg[agg['Method'] == method].iloc[0]
    values = [
        row['Storage Savings %'],
        row['Key Term Retention'],
        row['Recency Fidelity'],
        row['Info Density'],
    ]
    values += values[:1]
    lw = 3 if method == 'TSGC' else 1.5
    ax.plot(angles, values, 'o-', linewidth=lw, color=color, label=method)
    ax.fill(angles, values, alpha=alpha, color=color)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, size=11, color='#f0f2f8')
ax.set_ylim(0, 100)
ax.set_yticks([25, 50, 75, 100])
ax.set_yticklabels(['25', '50', '75', '100'], color=MUTED, size=8)
ax.grid(color='#2a2d3a', linestyle='--')
ax.spines['polar'].set_color('#2a2d3a')

legend = ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15),
                   facecolor='#1e2330', edgecolor='#2a2d3a',
                   labelcolor='#f0f2f8', fontsize=10)
ax.set_title('TSGC Radar — All Metrics', color='#f0f2f8', size=14, pad=20, fontweight='bold')

plt.tight_layout()
plt.savefig('tsgc_radar.png', dpi=150, bbox_inches='tight', facecolor='#0d0f14')
plt.show()
print('Saved: tsgc_radar.png')

In [ ]:
# ── FIGURE 3: Zone breakdown — what TSGC does per message ─────
conv = CONVERSATIONS[0]  # Use conv 1 (longest)
compressed = method_tsgc(conv)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('TSGC Zone Breakdown — How Each Message is Treated', fontsize=14, color='#f0f2f8', fontweight='bold')

# Left: per-message retention ratio
orig_lens = [len(m['content']) for m in conv]
comp_lens = [len(m['content']) for m in compressed]
zones     = [m.get('_zone', 1) for m in compressed]
zone_colors = {1: SUCCESS, 2: WARNING, 3: DANGER}
bar_colors  = [zone_colors[z] for z in zones]

x = range(len(compressed))
ax1.bar(x, orig_lens, color=MUTED, alpha=0.4, label='Original length', zorder=2)
ax1.bar(x, comp_lens, color=bar_colors, alpha=0.9, label='Compressed length', zorder=3)
ax1.set_xlabel('Message index (left = oldest)')
ax1.set_ylabel('Characters')
ax1.set_title('Per-message compression (by zone)')
ax1.grid(axis='y', zorder=0)
ax1.spines[['top','right']].set_visible(False)

patches = [
    mpatches.Patch(color=SUCCESS, label='Zone 1 — Verbatim (recent)'),
    mpatches.Patch(color=WARNING, label='Zone 2 — 40% kept'),
    mpatches.Patch(color=DANGER,  label='Zone 3 — 15% kept (old)'),
    mpatches.Patch(color=MUTED,   label='Original size'),
]
ax1.legend(handles=patches, fontsize=9, facecolor='#1e2330', edgecolor='#2a2d3a', labelcolor='#f0f2f8')

# Right: cumulative bytes saved across pipeline layers
raw_json    = json.dumps([{'role':m['role'],'content':m['content']} for m in conv]).encode()
dedup       = method_tsgc(conv)  # after layer 1+2
dedup_json  = json.dumps([{'role':m['role'],'content':m['content']} for m in dedup]).encode()
deflated    = len(zlib.compress(dedup_json, level=6))

layers = ['Raw', 'After\nDedup', 'After\nTemporal Prune', 'After\nDEFLATE (TSGC)']
sizes  = [len(raw_json), len(raw_json)*0.88, len(dedup_json), deflated]
lcolors= [MUTED, '#4285F4', WARNING, ACCENT]

bars = ax2.bar(layers, sizes, color=lcolors, width=0.5, edgecolor='none', zorder=3)
ax2.set_ylabel('Bytes stored')
ax2.set_title('Cumulative savings across TSGC pipeline')
ax2.grid(axis='y', zorder=0)
ax2.spines[['top','right']].set_visible(False)

for bar, val in zip(bars, sizes):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
             f'{int(val):,}B', ha='center', va='bottom', fontsize=9,
             color='#f0f2f8', fontweight='bold')

saving_pct = (1 - deflated / len(raw_json)) * 100
ax2.annotate(f'Total: {saving_pct:.1f}% saved',
             xy=(3, deflated), xytext=(2, len(raw_json)*0.6),
             fontsize=11, color=ACCENT, fontweight='bold',
             arrowprops=dict(arrowstyle='->', color=ACCENT))

plt.tight_layout()
plt.savefig('tsgc_pipeline.png', dpi=150, bbox_inches='tight', facecolor='#0d0f14')
plt.show()
print('Saved: tsgc_pipeline.png')

In [ ]:
# ── FIGURE 4: The key tradeoff — savings vs fidelity ──────────
fig, ax = plt.subplots(figsize=(10, 7))

for _, row in agg.iterrows():
    m   = row['Method']
    col = COLORS[m]
    sz  = 300 if m == 'TSGC' else 150
    ax.scatter(row['Storage Savings %'], row['Key Term Retention'],
               s=sz, color=col, zorder=5, edgecolors='white' if m=='TSGC' else 'none', linewidths=1.5)
    offset_x = 2 if m != 'RAW' else -3
    offset_y = 2 if m != 'UNIFORM-50%' else -4
    ax.annotate(m, (row['Storage Savings %'] + offset_x, row['Key Term Retention'] + offset_y),
                fontsize=11, color=col, fontweight='bold' if m=='TSGC' else 'normal')

# Ideal zone
ax.axhspan(80, 115, alpha=0.04, color=SUCCESS)
ax.axvspan(70, 115, alpha=0.04, color=SUCCESS)
ax.text(71, 81, '★ Ideal zone', fontsize=9, color=SUCCESS, alpha=0.7)

ax.set_xlabel('Storage Savings (%)')
ax.set_ylabel('Key Term Retention (%)')
ax.set_title('The Compression Tradeoff: Savings vs Semantic Fidelity', fontsize=13, fontweight='bold')
ax.set_xlim(-5, 110)
ax.set_ylim(40, 110)
ax.grid(zorder=0)
ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('tsgc_tradeoff.png', dpi=150, bbox_inches='tight', facecolor='#0d0f14')
plt.show()
print('Saved: tsgc_tradeoff.png')

## 6. Summary Table + References

In [ ]:
print('═' * 75)
print('TSGC BENCHMARK RESULTS — Average across 3 conversations')
print('═' * 75)
print(f'{"Method":<18} {"Savings":>10} {"KeyTerms":>12} {"Recency":>10} {"Density":>10}')
print('─' * 75)
for _, row in agg.iterrows():
    flag = ' ◄ TSGC' if row['Method'] == 'TSGC' else ''
    print(f"{row['Method']:<18} {row['Storage Savings %']:>9.1f}% {row['Key Term Retention']:>11.1f}% {row['Recency Fidelity']:>9.1f}% {row['Info Density']:>9.1f}%{flag}")
print('═' * 75)

print()
print('REFERENCES')
print('─' * 75)
refs = [
    '[1] Deutsch, P. (1996). DEFLATE Compressed Data Format Specification. RFC 1951.',
    '[2] Mihalcea, R. & Tarau, P. (2004). TextRank: Bringing Order into Texts. EMNLP.',
    '[3] Jiang, H. et al. (2023). LLMLingua: Compressing Prompts for Accelerated',
         '    Inference of Large Language Models. arXiv:2310.05736.',
    '[4] Reyna, V.F. & Brainerd, C.J. (1995). Fuzzy-trace theory: An interim synthesis.',
         '    Learning and Individual Differences. (Inspiration for temporal gradient)',
    '[5] Vanderwende, L. et al. (2007). Beyond SumBasic: Task-focused summarization',
         '    with sentence simplification and lexical expansion. IPM.',
]
for r in refs:
    print(r)